# jev-japanese-judgment Colabランチャー

このノートブックはリポジトリをcloneして依存をインストールするだけ。
実際のロジックは全てリポジトリ側のモジュールにある。
生成物・チェックポイントはGoogle DriveかHF Hubに保存すること。

**学習だけしたい場合**(データセットはHFにpush済み)は、env setup → Drive mount
→ HF_TOKEN → pullセル → trainセルの順で進め、データ変換/vLLM拡張/マージ/
リバランスのセルは全てスキップする。vLLMのセットアップは重い副作用
(torch関連パッケージの入れ替えによる破損リスク)があるので、本当に
LLM拡張をやり直す時だけ実行すること。

In [ ]:
!git clone https://github.com/fukayatti/jev-japanese-judgment.git
%cd jev-japanese-judgment
!pip install -q -r requirements.txt
# torchaoはColabに古いバージョン(0.10.0)がプリインストールされており、
# peft(学習で必ず使う)が要求する0.16.0以上と食い違ってImportErrorになる。
# vLLMとは無関係に、trainを使うセッションなら常に必要な修正。未使用なので外す。
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/jev-japanese-judgment-checkpoints'

In [ ]:
# Colabのユーザーシークレット(左メニューの鍵アイコン)に HF_TOKEN を登録しておくこと
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

## 途中から再開する場合

一度Hugging Faceにデータセットをpush済みなら、下のセルで`final_examples`を
直接取得できる。その場合、次のデータ変換〜マージまでの3セル(データ変換/
LLM拡張/マージ)は全てスキップして、学習のセルに進んでよい。

In [ ]:
from scripts.push_to_hub import pull

final_examples = pull(repo_id="fukayatti0/jev-japanese-judgment")
print(f'loaded from hub: {len(final_examples)} examples')

### 一度だけ: 公開済みデータのリバランス

JSNLI(train_w_filtering.tsv, 約53万件)がjcqa/chabsaに比べて桁違いに大きく、
既に公開したデータセットは97%がJSNLIで埋まってしまっていた。
vLLM/GPUは使わない(HFからのpull/pushとリスト操作のみ)ので、ここだけ実行すれば
9分の待ちは発生しない。直したら以降のセッションではこの節は不要。

In [ ]:
from data.postprocess.rebalance import subsample_by_source
from data.postprocess.shuffle_candidates import shuffle_all
from scripts.push_to_hub import pull, push

_current = pull(repo_id="fukayatti0/jev-japanese-judgment")
print(f'before rebalance: {len(_current)}')

_balanced = subsample_by_source(_current, max_per_source=10000)
final_examples = shuffle_all(_balanced)
print(f'after rebalance: {len(final_examples)}')

push(final_examples, repo_id="fukayatti0/jev-japanese-judgment", private=False)
print('pushed rebalanced dataset')

In [ ]:
import subprocess
from pathlib import Path

from data.convert import chabsa, jcommonsenseqa, jsnli

# JCommonsenseQA / chABSA は HF Hub (parquet) から直接ロードできる
jcqa_examples = jcommonsenseqa.convert('train')
chabsa_examples = chabsa.convert('train')

# JSNLIは配布形式がzipなのでダウンロード・展開してからパスを渡す。
# train_w_filtering.tsv は JSNLI公式の学習用フルセットで約533,000件ある
# (jcqa/chabsaの数万件規模とは桁違いに大きい。バランスは後段のマージセルで取る)。
jsnli_dir = Path('data/raw/jsnli')
jsnli_dir.mkdir(parents=True, exist_ok=True)
if not (jsnli_dir / 'jsnli_1.1' / 'train_w_filtering.tsv').exists():
    subprocess.run(['curl', '-sL', '-o', str(jsnli_dir / 'jsnli.zip'),
                     'https://nlp.ist.i.kyoto-u.ac.jp/nl-resource/JSNLI/jsnli_1.1.zip'], check=True)
    subprocess.run(['unzip', '-o', '-q', str(jsnli_dir / 'jsnli.zip'), '-d', str(jsnli_dir)], check=True)

jsnli_examples = jsnli.convert(jsnli_dir / 'jsnli_1.1' / 'train_w_filtering.tsv')

all_examples = jcqa_examples + chabsa_examples + jsnli_examples
print(f'jcqa={len(jcqa_examples)} chabsa={len(chabsa_examples)} jsnli={len(jsnli_examples)} total={len(all_examples)}')

### LLM拡張をやる場合だけ: vLLMのセットアップ

学習だけ(pullしてtrainだけ)なら、このセルは実行しないこと。vLLMは
torchを別のCUDAビルドに上げてしまい、その後始末(torchaudio/torchao削除、
torchvision入れ直し)で同一セッション内でtorch関連パッケージを何度も
いじることになり、`libtorch_python.so: cannot open shared object file`
のようなtorchのネイティブライブラリ破損を引き起こすことがある
(実際に発生した)。LLM拡張をやり直さない限り、vLLMは不要。

In [ ]:
!pip install -q vllm
# vllmがtorchを別のCUDAビルドに上げてしまい、Colab既存のtorchaudioとCUDA
# バージョンが食い違ってRuntimeErrorになることがある。torchaudioは未使用なので外す。
# torchvisionは一度外すとQwen3.5(VLモデル)のロード自体がtorchvisionをimportして
# いるため ModuleNotFoundError になる。torchvisionは削除せず、vllmインストール後の
# torchに合わせて入れ直す(pipの依存解決に現在のtorchを見てもらう)。
# (torchaoの修正はcell-1で既に実施済み。ここではvLLM固有の問題だけ扱う)
!pip uninstall -y -q torchaudio
!pip install -q -U torchvision

In [ ]:
from data.augment.generate import build_prompts, run_batch_generation

# vLLM本体はsubprocess (data/augment/vllm_worker.py) 側で実行する。
# ノートブックのカーネル内で直接vllm.LLM(...)を呼ぶと、CUDA初期化済みの
# プロセスからspawnしようとしてデッドロックすることがあるため。
#
# エンジン初期化だけで約9分かかる(T4でのTritonカーネルJITコンパイル)ので、
# 複数回に分けず一気に実行すること。all_examples全件(jsnliだけで53万件超)を
# 拡張するのは非現実的なので、先頭の一部だけ拡張する。件数を増やしたい場合は
# この3000という数字を変えること。
AUGMENT_TARGET_SIZE = 3000
jobs = build_prompts(all_examples[:AUGMENT_TARGET_SIZE])
results = run_batch_generation(jobs)
for r in results[:5]:
    print(r['kind'], '->', r['output'])

In [ ]:
from data.augment.merge import merge_augmentations
from data.postprocess.rebalance import subsample_by_source
from data.postprocess.shuffle_candidates import shuffle_all

augmented_examples = merge_augmentations(all_examples, results)
print(f'augmented examples created: {len(augmented_examples)}')

# JSNLIだけ桁違いに大きい(53万件超)ので、source_datasetごとに上限を設けて
# バランスを取ってから、候補の順序をシャッフルし labelインデックスを再計算する。
combined = subsample_by_source(all_examples + augmented_examples, max_per_source=10000)
final_examples = shuffle_all(combined)
print(f'final_examples total: {len(final_examples)}')

In [ ]:
from train import main as train_main

# checkpoint_dirを渡すと、Driveに200ステップごと+各エポック終了時に
# 学習対象(LoRA+ヘッド)だけ保存される。セッションが切れても学習を再開できる。
model = train_main(final_examples, checkpoint_dir=CHECKPOINT_DIR)

### セッションが切れて学習が中断した場合

上のtrainセルの代わりに、このセルを実行する。`"latest"`は200ステップごとに
上書き保存されている最新のチェックポイント、`"epoch1"`のようにepoch番号
指定も可能。optimizerの状態(運動量)は保存されないため完全に同じ続きには
ならないが、LoRA+ヘッドの学習済み重みは引き継がれる。

In [ ]:
from train import main as train_main

model = train_main(
    final_examples,
    checkpoint_dir=CHECKPOINT_DIR,
    resume_from="latest",  # または "epoch1" など
)

In [ ]:
from scripts.push_to_hub import push

# HF_TOKENは前段のセルで環境変数にセット済み。ここではrepo_idのみ指定する
push(final_examples, repo_id="fukayatti0/jev-japanese-judgment", private=False)

## 最終評価とモデル公開

3エポック比較の結果、epoch1が最良(以降は過学習気味で精度が伸びない)と
確認済み。学習に使っていない分割(未使用データ)でaccuracy/ECE/Brier/NLLを測り直し、その結果を
モデルカードに埋め込んでHugging Faceに公開する。

In [ ]:
!git pull
from scripts.build_clean_eval import build, load, save, _metrics
from eval.run_eval import load_model_from_checkpoint
from scripts.push_model_to_hub import push_model
from pathlib import Path

# 学習に使っていない分割(JCommonsenseQA validation / chABSA test / JSNLI dev)で評価する。
# train_val_split(final_examples)は学習データと重なるので、公開する数値には使わない。
clean_path = Path('data/clean_eval.jsonl')
if not clean_path.exists():
    save(build(), clean_path)
clean_examples = load(clean_path)

eval_model = load_model_from_checkpoint(CHECKPOINT_DIR, 'epoch1')
eval_results = _metrics(eval_model, clean_examples)
print(eval_results)

push_model(
    checkpoint_dir=CHECKPOINT_DIR,
    tag='epoch1',
    repo_id='fukayatti0/jev-japanese-judgment',
    eval_results=eval_results,
    private=False,
)
print('pushed model -> https://huggingface.co/fukayatti0/jev-japanese-judgment')
# 注意: push_modelはREADME.mdをモデルカードのテンプレートで作り直す。量子化版/GGUF版の節も
# 消えるので、実行後は `python -m scripts.update_model_card` でカード全体を再生成すること。


## 追加学習: noul(yes/no)とscore(段階評価)

JevBenchのnoul型でyes/noに弱かった(fact 58%)ので、JNLI/JCoLA(yes/no)とJSTS/JSICK(段階)を、既存3タスクを少量混ぜて
epoch1から追加学習する。元のチェックポイントは上書きしない(`-v2`の別ディレクトリに保存)。
評価は学習に使っていない分割(JNLI test / JCoLA valid / JSTS validation / JSICK test)。学習の前後で比べる。


In [ ]:
# 1. ベースライン(追加学習の前): epoch1を、noul/scoreの未使用データと、従来の未使用データで測る
!git pull
!python -m scripts.build_clean_eval build-extras eval-ckpt --out data/clean_eval_extras.jsonl --checkpoint-dir $CHECKPOINT_DIR --tag epoch1
!python -m scripts.build_clean_eval build eval-ckpt --out data/clean_eval.jsonl --checkpoint-dir $CHECKPOINT_DIR --tag epoch1


In [ ]:
# 2. 追加学習(epoch1の重みから、新しい学習として1エポック)
from scripts.build_judgment_mix import build_mix
from scripts.push_to_hub import pull
from train import main as train_main

mix = build_mix(pull(repo_id="fukayatti0/jev-japanese-judgment"))
NEW_CKPT_DIR = CHECKPOINT_DIR + "-v2"
model = train_main(
    mix,
    epochs=1,
    lr=1e-4,
    checkpoint_dir=NEW_CKPT_DIR,
    init_from="epoch1",
    init_checkpoint_dir=CHECKPOINT_DIR,
)


In [ ]:
# 3. 追加学習の後: 同じ評価セットで測り直す
!python -m scripts.build_clean_eval eval-ckpt --out data/clean_eval_extras.jsonl --checkpoint-dir $NEW_CKPT_DIR --tag epoch1
!python -m scripts.build_clean_eval eval-ckpt --out data/clean_eval.jsonl --checkpoint-dir $NEW_CKPT_DIR --tag epoch1
